## Pyspark intro
Pyspark is the Python API to Apache Spark

a short note on notebooks in databricks - can choose between
- SQL notebook
- Python notebook

the choise all cells default to that choice e.g SQL -> cells become SQL by default

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"
#/Volumes/data/olympic_games/raw_data/noc_regions.csv

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_athletes

In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
(df_athletes.count(), len(df_athletes.columns))

# Infer the schema

we note that Age, height becomes strings, that is because it has null values

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)
df_athletes

## Define explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

In [0]:
df_regions = spark.read.csv(f"{DATA_PATH}/noc_regions.csv", header=True)
df_regions.display()


## EDA

PySpark transformations

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE','NOR', 'DEN', 'FIN', 'ISL') AND Medal != 'NA'"
).sort("NOC").display()

Spark SQL

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

# TODO: pick out swidish medels in table tennis

spark.sql(
    "SELECT (*) FROM df_athletes_schema WHERE NOC = 'SWE' AND Sport = 'Table Tennis' AND medal != 'NA'"
).display()

## Count medals and plot

In [0]:
df_swe_medals = spark.sql("""
          SELECT 
          sport, COUNT(medal) AS medal_count
          FROM df_athletes_schema
          WHERE NOC = 'SWE' AND medal IN ('Bronze', 'Silver', 'Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind="bar", y="sport", x="medal_count")
fig.update_layout(yaxis = {"autorange": "reversed"})

## Ingesting data to Unity Catalog

In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals AS
(
  SELECT
    name,
    age,
    height,
    weight,
    year,
    sport,
    medal
  FROM
    df_athletes_schema
  WHERE
    noc = 'SWE'
    and medal IN ('Bronze', 'Silver', 'Gold')
);

In [0]:
%sql
SELECT name, sport
FROM data.olympic_games.sweden_medals
WHERE sport = 'Swimming'
ORDER BY name;